# IS 362 – Week 5 Assignment
**Questions**
1. What is the northernmost airport in the United States?
2. What is the easternmost airport in the United States?
3. On February 12th, 2013, which New York area airport had the windiest weather?

**Data:** CSV files from https://github.com/hadley/nycflights13/tree/master/data-raw (`airports.csv`, `weather.csv`).

In [17]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/hadley/nycflights13/master/data-raw/"
# If these URLs fail, download the CSVs from the repo and use local paths instead,
# e.g. airports = pd.read_csv("airports.csv")
airports = pd.read_csv(BASE + "airports.csv")
weather  = pd.read_csv(BASE + "weather.csv")

print(airports.shape, weather.shape)
airports.head()

(1458, 8) (26115, 15)


,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


## Data corrections (found while researching)
Some rows in `airports.csv` contain errors, so I corrected them before answering Q1 and Q2:
- **EEN** (Dillant Hopkins Airport, Keene, NH): latitude and longitude were swapped and the longitude sign was lost (listed as 72.27, +42.90). Corrected to 42.898333, -72.270833.
- **MYF** (Montgomery Field, San Diego, CA) and **DVT** (Deer Valley, Phoenix, AZ): longitude had the wrong sign (positive instead of negative). Corrected by making it negative.

In [18]:
airports.loc[airports["faa"]=="EEN", ["lat","lon"]] = [42.898333, -72.270833]
airports.loc[airports["faa"].isin(["MYF","DVT"]), "lon"] = -airports.loc[airports["faa"].isin(["MYF","DVT"]), "lon"].abs()
airports[airports["faa"].isin(["EEN","MYF","DVT"])][["faa","name","lat","lon"]]

,faa,name,lat,lon
396,DVT,Deer Valley Municipal Airport,33.411700,-112.457000
417,EEN,Dillant Hopkins Airport,42.898333,-72.270833
942,MYF,Montgomery Field,32.475900,-117.759000


## Question 1 – Northernmost airport
**Approach:** sort by latitude (`lat`) descending and look at the top five candidates.

In [19]:
airports.sort_values("lat", ascending=False).head(5)[["faa","name","lat","lon"]]

,faa,name,lat,lon
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003
110,AIN,Wainwright Airport,70.638056,-159.994722
708,K03,Wainwright As,70.613378,-159.860350
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500


### Verification (Q1)
*Fill in after running:* the top row should be Wiley Post–Will Rogers Memorial Airport (Utqiaġvik/Barrow, Alaska, FAA code BRW/BRW), around 71.29°N. Confirm with an outside source (FAA / Wikipedia / a map – the map in the assignment PDF shows this airport on the Arctic Ocean coast) and note the source URL here.

**Answer 1:** _Wiley Post–Will Rogers Memorial Airport (Utqiaġvik, AK) – confirm with your output._

## Question 2 – Easternmost airport
**Approach:** "East" means a larger longitude. Sort by `lon` descending and look at the top five.

In [20]:
airports.sort_values("lon", ascending=False).head(5)[["faa","name","lat","lon"]]

,faa,name,lat,lon
1290,SYA,Eareckson As,52.712275,174.113620
444,EPM,Eastport Municipal Airport,44.910111,-67.012694
624,HUL,Houlton Intl,46.123083,-67.792056
259,CAR,Caribou Muni,46.871500,-68.017917
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797


### Data issue to watch for
Longitude is negative in the Western Hemisphere. Alaska's Aleutian Islands cross the 180° meridian, so airports such as Shemya (Eareckson Air Station) have a **positive** longitude (~+174°) and sort to the top – but they are on the *far western* edge of the U.S. in the usual sense. Likewise Guam has positive longitude. Decide how to treat these and **document the decision**. A reasonable approach: restrict to airports with negative longitude (Western Hemisphere) and then look at the top five.

In [21]:
west = airports[airports["lon"] < 0]
east_top5 = west.sort_values("lon", ascending=False).head(5)[["faa","name","lat","lon"]]
east_top5

,faa,name,lat,lon
444,EPM,Eastport Municipal Airport,44.910111,-67.012694
624,HUL,Houlton Intl,46.123083,-67.792056
259,CAR,Caribou Muni,46.871500,-68.017917
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797
1398,WFK,Northern Aroostook Regional Airport,47.285556,-68.312778


### Verification (Q2)
*Fill in after running:* search for the top candidate(s) (e.g., Eastport, Maine / Lubec, Maine area, or any Puerto Rico / U.S. Virgin Islands airport that appears – decide whether territories count) and confirm the location with a second source. Record your sources and reasoning here.

**Answer 2:** _(state the airport and your assumption about Aleutians/territories)_

## Question 3 – Windiest NY-area airport on Feb 12, 2013

In [22]:
feb12 = weather[(weather["year"]==2013)&(weather["month"]==2)&(weather["day"]==12)]
feb12.groupby("origin")["wind_speed"].agg(["count","mean","max"])

,count,mean,max
origin,,,
EWR,24,56.38822,1048.36058
JFK,24,14.38475,20.71404
LGA,24,14.96014,23.01560


### Check for outliers
Wind speeds in this dataset are in mph. Look at the raw maxima and the distribution.

In [23]:
feb12.sort_values("wind_speed", ascending=False).head(10)[["origin","hour","wind_speed","wind_gust","wind_dir"]]

,origin,hour,wind_speed,wind_gust,wind_dir
1009,EWR,3,1048.36058,NaN,260.0
18417,LGA,2,23.01560,31.07106,290.0
1018,EWR,12,21.86482,31.07106,270.0
18428,LGA,13,21.86482,25.31716,300.0
9717,JFK,8,20.71404,27.61872,280.0
1008,EWR,2,20.71404,25.31716,270.0
18429,LGA,14,20.71404,25.31716,300.0
18416,LGA,1,19.56326,28.76950,270.0
9716,JFK,7,19.56326,NaN,270.0
18427,LGA,12,19.56326,26.46794,270.0


A wind speed of over 1,000 mph is physically impossible (the strongest surface wind ever recorded is ~253 mph), so a value like that is a data error. I treat readings above a plausible threshold (e.g., 100 mph) as invalid and exclude them.

In [24]:
clean = feb12[feb12["wind_speed"] <= 100]
removed = feb12[feb12["wind_speed"] > 100]
print("Rows removed:"); display(removed[["origin","hour","wind_speed"]])

summary = clean.groupby("origin")["wind_speed"].agg(["count","mean","max"]).sort_values("mean", ascending=False)
summary

Rows removed:


,origin,hour,wind_speed
1009,EWR,3,1048.36058


,count,mean,max
origin,,,
LGA,24,14.960140,23.01560
JFK,24,14.384750,20.71404
EWR,23,13.258987,21.86482


### Answer 3
*Fill in after running:* state the airport with the highest average (and highest max) cleaned wind speed, and note whether the answer differs depending on mean vs. max. Mention that the outlier row(s) removed above would otherwise have made that airport appear windiest.

## Assumptions and data changes (summary)
- Longitude sign convention / Aleutian & territory handling (Q2): _describe_
- Removed wind-speed readings > 100 mph as sensor/data errors (Q3): _list rows_
- Sources used for verification: _links_